In [20]:
# Change from boolean types to string

recalls["UAL_FLAG"] = recalls["UAL_FLAG"].map({True: "TRUE",False:"FALSE"})
recalls["DETERMINATE_FLAG"] = recalls["DETERMINATE_FLAG"].map({True: "True",False:"False"})

In [21]:
#Deceased? Not UAL but have no RTC date -LEAVE IT AS IT IS
retain = ['PRISON_NUMBER','LICENCE_REVOKE_DATE', 'RTC_DATE', 'UAL_FLAG', 'RECALL_TYPE_DESCRIPTION']

check1 = recalls[(recalls["RTC_DATE"].isna()) & (recalls["UAL_FLAG"] != "TRUE")][retain]
check1.shape

(0, 5)

In [22]:
# UAL but have RTC date?

retain = ['PRISON_NUMBER','LICENCE_REVOKE_DATE', 'RTC_DATE', 'UAL_FLAG', 'RECALL_TYPE_DESCRIPTION']
check1 = recalls[(recalls["RTC_DATE"].notna()) & (recalls["UAL_FLAG"] == "TRUE")][retain]
check1.shape

(0, 5)

In [23]:
# Rename column
recalls.rename(columns = {'CUSTODY_TYPE_AT_TIME_OF_RECALL_DESCRIPTION':'CUSTODY_TYPE_AT_RECALL'}, 
               inplace = True)


# Replace Not Applicable and Not Specified custody types
Unknowns = ['Not Applicable','Not Specified']

recalls['CUSTODY_TYPE_AT_RECALL'] = np.where(recalls['CUSTODY_TYPE_AT_RECALL'].isin(Unknowns), 
                                             recalls['CUSTODY_TYPE_DESCRIPTION'],
                                             recalls['CUSTODY_TYPE_AT_RECALL'])

In [24]:
# Match Extras onto original

recalls_pub = duckdb.sql("""
    SELECT a.*, b.DOS, b.NUMBER_OF_RECALL_REASONS, 
        b.RECALL_REASON_DESCRIPTIONS,b.TARIFF_EXPIRY_DATE 
    FROM recalls AS a LEFT JOIN extras AS b 
    ON  
        (a.PRISON_NUMBER = b.PRISON_NUMBER  OR
         a.FILE_REFERENCE = b.FILE_REFERENCE OR
         a.NOMS_ID = b.NOMS_ID
        ) AND 
        a.LICENCE_REVOKE_DATE = b.LICENCE_REVOKE_DATE""").df()

recalls_pub.shape

(6209, 40)

In [25]:
# Import Look up tables

CustodyRef = pd.read_csv('/home/jovyan/OMPPG/Recalls/Reference Data/Recalls Lookup/CustodyType.csv')
AreaRef = pd.read_csv('/home/jovyan/OMPPG/Recalls/Reference Data/Recalls Lookup/Probation_Areas.csv')
RecallRef = pd.read_csv('/home/jovyan/OMPPG/Recalls/Reference Data/Recalls Lookup/RecallType.csv')

In [322]:
# PrisonerRef = pd.read_csv('/home/jovyan/OMPPG/Recalls/Reference Data/Recalls Lookup/Some_Prisoners.csv')

#PrisonerRef = dateTimeConvert(PrisonerRef, "LICENCE_REVOKE_TIME")
#PrisonerRef = dateTimeConvert(PrisonerRef, "RTC_TIME")

# Add any offender-specific information*/
# Split lookup into individual updates

# Area_Update = duckdb.sql("\
#     SELECT DISTINCT PROBATION_AREA_DESCRIPTION, PRISON_NUMBER, LICENCE_REVOKE_TIME\
#     FROM PrisonerRef\
#     WHERE PROBATION_AREA_DESCRIPTION IS NOT NULL").df()


# RTC_Update = duckdb.sql("\
#     SELECT DISTINCT RTC_TIME, PRISON_NUMBER, LICENCE_REVOKE_TIME\
#     FROM PrisonerRef\
#     WHERE RTC_TIME IS NOT NULL").df()

# RTC_Update

# Gender_Update = duckdb.sql("\
#     SELECT DISTINCT PRISON_NUMBER, GENDER\
#     FROM PrisonerRef\
#     WHERE GENDER IS NOT NULL").df()

# Gender_Update

In [34]:
# KEY********************Check non-Matches and upadate reference files and run again**************

Recalls_Unmatched_Codes = duckdb.sql("""
        SELECT a.PRISON_NUMBER, a.CUSTODY_TYPE_AT_RECALL, c.custodytype, c.custodytype2,
               a.PROBATION_AREA_DESCRIPTION,a.NOMS_REGION_DESCRIPTION, d.prob_area, d.SUP_BODY, d.NPS_CRC_NAME, d.NPS_DIVISION,
               a.RECALL_TYPE_DESCRIPTION, e.targettype, e.sentencetype
        FROM recalls_pub AS a 
        LEFT JOIN CustodyRef AS c
            ON (a.CUSTODY_TYPE_AT_RECALL = C.CUSTODY_TYPE_AT_RECALL)
        LEFT JOIN AreaRef AS d
            ON (a.PROBATION_AREA_DESCRIPTION = d.PROBATION_AREA_DESCRIPTION)
        LEFT JOIN RecallRef AS e
            ON (a.RECALL_TYPE_DESCRIPTION = e.Recall_Type_Description)
        WHERE (PROB_AREA IS NULL
            OR SUP_BODY IS NULL
            OR targettype IS NULL
            OR custodytype IS NULL
            OR custodytype2 IS NULL
            OR sentencetype IS NULL)
            AND a.PROBATION_AREA_DESCRIPTION NOT IN ('Scotland','Not Applicable','Northern Ireland','Channel Islands')
            AND a.RECALL_TYPE_DESCRIPTION != 'Not Applicable'
            AND a.CUSTODY_TYPE_AT_RECALL != 'Not Applicable' """).df()

Recalls_Unmatched_Codes.shape

(0, 13)

In [30]:
Recalls_Matched = duckdb.sql("""
        SELECT a.*, c.custodytype, c.custodytype2, d.prob_area, d.SUP_BODY, d.NPS_CRC_NAME, d.NPS_DIVISION,
               e.targettype, e.sentencetype
        FROM recalls_pub AS a 
        LEFT JOIN CustodyRef AS c
            ON (a.CUSTODY_TYPE_AT_RECALL = C.CUSTODY_TYPE_AT_RECALL)
        LEFT JOIN AreaRef AS d
            ON (a.PROBATION_AREA_DESCRIPTION = d.PROBATION_AREA_DESCRIPTION)
        LEFT JOIN RecallRef AS e
            ON (a.RECALL_TYPE_DESCRIPTION = e.Recall_Type_Description)
        WHERE (a.PROBATION_AREA_DESCRIPTION != 'Scotland' AND
               a.PROBATION_AREA_DESCRIPTION != 'Northern Ireland' AND
               a.PROBATION_AREA_DESCRIPTION != 'Channel Islands') """).df()

Recalls_Matched.shape

(6209, 48)

In [31]:
Problem_Days = [(31,1),(31,3),(30,6),(30,9),(31,12)]

Recalls_Matched["return_by"] =  Recalls_Matched["LICENCE_REVOKE_DATE"].apply(lambda x: x + pd.tseries.offsets.QuarterEnd()*(1 if (x.day,x.month) in Problem_Days else 2))

diffcond = Recalls_Matched['RTC_TIME'].notna() & Recalls_Matched['PB_DECISION_AFTER_BREACH_ACTUAL'].notna()
Recalls_Matched["diff"] = np.where(diffcond, (Recalls_Matched['RTC_TIME'] - Recalls_Matched['PB_DECISION_AFTER_BREACH_ACTUAL']).dt.total_seconds(),np.nan)

Pincond = Recalls_Matched['REPORT_RECD_BY_UNIT_ACTUAL'].notna() & Recalls_Matched['REPORT_RECD_BY_UNIT_TARGET'].notna() & (Recalls_Matched['REPORT_RECD_BY_UNIT_ACTUAL'] <=  Recalls_Matched['REPORT_RECD_BY_UNIT_TARGET']) 
Recalls_Matched["Pintarget"] = np.where(Pincond, 1,np.nan)

PPUcond = Recalls_Matched['LICENCE_REVOKE_TIME'].notna() & Recalls_Matched['ISSUE_REVOCATION_TARGET'].notna() & (Recalls_Matched['LICENCE_REVOKE_TIME'] <=  Recalls_Matched['ISSUE_REVOCATION_TARGET']) 
Recalls_Matched["PPUintarget"] = np.where(PPUcond, 1,np.nan)

Polcond = Recalls_Matched['RTC_TIME'].notna() & Recalls_Matched['RTC_TARGET'].notna() & (Recalls_Matched['RTC_TIME'] <=  Recalls_Matched['RTC_TARGET']) 
Recalls_Matched["Policeintarget"] = np.where(Polcond, 1,np.nan)

Standcond = Recalls_Matched['diff'].notna() & (Recalls_Matched['diff'] < 144*60*60) & (Recalls_Matched['targettype'] == "standard") 
Recalls_Matched["standardtarget"] = np.where(Standcond, 1,np.nan)

Emergcond = Recalls_Matched['diff'].notna() & (Recalls_Matched['diff'] < 74*60*60) & (Recalls_Matched['targettype'] == "emergency") 
Recalls_Matched["emergencytarget"] = np.where(Emergcond, 1,np.nan)

In [32]:
def RS_func(df):
    if  (df["targettype"] == 'standard') & (df["UAL_FLAG"] == 'FALSE') & ((pd.isnull(df["RTC_DATE"])) | (pd.isnull(df["diff"]))):
        return 'Standard - Missing data'
    elif (df["targettype"] == 'standard') & ((df["UAL_FLAG"] =='TRUE') | ((df["LICENCE_REVOKE_DATE"].year < 2015) & (df["RTC_DATE"] > df["return_by"]))):
        return 'Standard - Not returned'
    elif (df["targettype"] == 'standard') & (df["diff"] < 144*60*60):
        return 'Standard - Less than 144 hours'
    elif (df["targettype"] == 'standard') & (df["diff"] >= 144*60*60):
        return 'Standard - 144 hours to 3 months'
    elif (df["targettype"] == 'emergency') & (df["UAL_FLAG"] =='FALSE') & ((pd.isnull(df["RTC_DATE"])) | (pd.isnull(df["diff"]))):
        return 'Emergency - Missing data'
    elif (df["targettype"] == 'emergency') & ((df["UAL_FLAG"] =='TRUE') | ((df["LICENCE_REVOKE_DATE"].year < 2015) & (df["RTC_DATE"] > df["return_by"]))):
        return 'Emergency - Not returned'
    elif (df["targettype"] == 'emergency') & (df["diff"] < 74*60*60):
        return 'Emergency - Less than 74 hours';
    elif (df["targettype"] == 'emergency') & (df["diff"] >= 74*60*60): 
        return 'Emergency - 74 hours to 3 months'

Recalls_Matched["recall_status"] = Recalls_Matched.apply(RS_func,axis = 1)

In [33]:
RecLenCond = Recalls_Matched['RECALL_TYPE_DESCRIPTION'].str.contains("fix|ftr",case=False)
Recalls_Matched["Recall_Length"] = np.where(RecLenCond,'Fixed Term','Standard')


RecProcCond = Recalls_Matched['recall_status'].str.contains("Emergency",case=False)
Recalls_Matched["recall_process"] = np.where(RecProcCond,'Emergency','Standard')

Recalls_Matched.loc[Recalls_Matched["custodytype2"] =='b. indeterminate', "Recall_Length"] = 'Standard'
Recalls_Matched.loc[Recalls_Matched["custodytype2"] =='b. indeterminate', "recall_process"] = 'Emergency'
Recalls_Matched.loc[Recalls_Matched["custodytype2"] =='b. indeterminate', "sentencetype"] = 'Other'

Recalls_Matched.loc[Recalls_Matched["recall_process"] =='Emergency', "recall_length"] = 'Standard'
Recalls_Matched.loc[Recalls_Matched["LICENCE_REVOKE_DATE"] < pd.Timestamp(year=2015, month=2, day=1), "sentencetype"] = 'Other'

InTarget = ['Standard - Less than 144 hours','Emergency - Less than 74 hours']
OutTarget =['Standard - 144 hours to 3 months','Emergency - 74 hours to 3 months']
NoReturn = ['Standard - Not returned','Emergency - Not returned']

Recalls_Matched["recall_target"] = np.where(Recalls_Matched["recall_status"].isin(InTarget), 'a. Returned in target',
                                       np.where(Recalls_Matched["recall_status"].isin(OutTarget),'b. Returned outside target',
                                           np.where(Recalls_Matched["recall_status"].isin(NoReturn),' c. Not returned','d. Resolved') 
                                               ) 
                                           )

Recalls_Matched["further_charge"] = np.where(Recalls_Matched['RECALL_REASON_DESCRIPTIONS'].str.contains("Further",case=False),100,0)
Recalls_Matched.loc[Recalls_Matched["LICENCE_REVOKE_DATE"] < pd.Timestamp(year=2014, month=6, day=1), "SUP_BODY"] = 'a. PT'

Recalls_Matched.loc[Recalls_Matched["GENDER"] == 'F ( Was M )', "GENDER"] = 'F'
Recalls_Matched.loc[Recalls_Matched["GENDER"] == 'M ( Was F )', "GENDER"] = 'M'

In [34]:
# De-duplicate dataset

recalls_nodup = Recalls_Matched.copy()
recalls_nodup.sort_values(['PRISON_NUMBER','LICENCE_REVOKE_DATE'], inplace=True)
recalls_nodup = recalls_nodup.drop_duplicates(subset=['PRISON_NUMBER','LICENCE_REVOKE_DATE'])

In [ ]:
# Export


In [2]:
to_test = "GENDER"
Recalls_Matched.groupby(to_test)[to_test].count()

NameError: name 'Recalls_Matched' is not defined